# 52. Progress Bars with tqdm — `tqdm`
**Python Libraries Lab · Phase 3 · Utilities & Testing**

_Know how long things will take, with one wrapper._

✅ **Runs in Colab** as-is.

When a loop takes minutes, you want to know whether it is working and how long is left. tqdm adds a progress bar by wrapping any loop, showing how far along it is, how fast it is going, and an estimated time to finish. The name comes from the Arabic word taqaddum, which means progress.

**In this notebook:** You will add progress bars to ShopEasy jobs. Processing orders, downloading files in chunks with a manual counter, nested bars for months and days, a pandas progress apply, and a parallel job with a thread pool.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q tqdm
import importlib.metadata as md
for pkg in ['tqdm']:
    print(pkg, md.version(pkg))

## Step 1 · Wrap any loop
`tqdm.auto` chooses a notebook or terminal bar automatically.

In [ ]:
from tqdm.auto import tqdm, trange
import time

total = 0
for order_id in tqdm(range(1001, 1101), desc="Processing orders", unit="order"):
    time.sleep(0.01)
    total += 1
print("processed", total)

## Step 2 · Manual updates
When work arrives in chunks, update the bar yourself. `unit_scale` shows KB/MB.

In [ ]:
file_size = 5_000_000
with tqdm(total=file_size, unit="B", unit_scale=True, desc="Downloading catalogue.csv") as bar:
    done = 0
    while done < file_size:
        chunk = min(250_000, file_size - done)
        time.sleep(0.02)
        done += chunk
        bar.update(chunk)

## Step 3 · Nested bars and messages
`leave=False` removes the inner bar when finished. `tqdm.write` prints without breaking bars.

In [ ]:
for month in tqdm(["Jan", "Feb", "Mar"], desc="Months"):
    for day in trange(10, desc=f"{month} days", leave=False):
        time.sleep(0.01)
    tqdm.write(f"{month} report saved")

## Step 4 · pandas progress_apply
`tqdm.pandas()` adds `.progress_apply` to DataFrames and Series.

In [ ]:
import pandas as pd
tqdm.pandas(desc="Scoring customers")
df = pd.DataFrame({"customer": range(300), "orders": [i % 7 for i in range(300)]})

def loyalty_score(n):
    time.sleep(0.002)
    return "gold" if n >= 5 else "silver" if n >= 3 else "bronze"

df["tier"] = df["orders"].progress_apply(loyalty_score)
print(df["tier"].value_counts())

## Step 5 · Parallel jobs with a progress bar
Threads do the work; `as_completed` updates the bar as each finishes.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
def send_invoice(i):
    time.sleep(0.2)
    return i

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = [pool.submit(send_invoice, i) for i in range(40)]
    sent = [f.result() for f in tqdm(as_completed(futures), total=len(futures), desc="Sending invoices")]
print(len(sent), "invoices sent")

## ✍️ Your turn
Use `tqdm` to loop over `range(50)` and add up the numbers. Store the **sum** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 1225, f"Expected 1225, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = 0
for i in tqdm(range(50), desc="Adding"):
    answer += i

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **tqdm(iterable)**
- **tqdm.auto**
- **ETA**
- **Manual update**
- **progress_apply**
- **Nested bars**

**Next:** 53. Settings & Secrets (`python-dotenv`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/53_python_dotenv.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install tqdm
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.